# 02 Preprocessing and EDA

Goal: clean anomalous categories, save the interim dataset, and produce the first EDA figures.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_preprocessing import (
    load_raw_dataset,
    clean_dataset,
    data_quality_report,
    save_json,
    RAW_DATA_PATH,
    INTERIM_DATA_PATH,
    TARGET_COLUMN,
)

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


In [ ]:
raw_df = load_raw_dataset(RAW_DATA_PATH)
clean_df = clean_dataset(raw_df, drop_duplicate_rows=False)
clean_df.to_csv(INTERIM_DATA_PATH, index=False)

save_json(
    {
        "before_cleaning": data_quality_report(raw_df),
        "after_cleaning": data_quality_report(clean_df),
    },
    TABLES_DIR / "data_quality_report.json",
)

clean_df.shape

In [ ]:
target_counts = clean_df[TARGET_COLUMN].value_counts().sort_index()
ax = target_counts.rename({0: "Non-default", 1: "Default"}).plot(kind="bar", color=["#2a9d8f", "#e76f51"])
ax.set_title("Target Class Distribution")
ax.set_xlabel("")
ax.set_ylabel("Number of customers")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "target_class_distribution.png", dpi=300)
plt.show()


In [ ]:
pay0_default_rate = clean_df.groupby("PAY_0")[TARGET_COLUMN].mean().reset_index()
plt.figure(figsize=(8, 4))
sns.barplot(data=pay0_default_rate, x="PAY_0", y=TARGET_COLUMN, color="#457b9d")
plt.title("Default Rate by PAY_0")
plt.xlabel("PAY_0 repayment status")
plt.ylabel("Default rate")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pay0_default_rate.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
corr = clean_df.drop(columns=["ID"], errors="ignore").corr(numeric_only=True)
sns.heatmap(corr, cmap="vlag", center=0, linewidths=0.1)
plt.title("Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=300)
plt.show()
